<a href="https://colab.research.google.com/github/harinijk/PredictPrice/blob/main/CSCI_4521_Hw3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch


np.random.seed(42)
torch.manual_seed(42)

Question 1

In [ ]:
!wget https://raw.githubusercontent.com/SJGuy-UMN/CSCI4521/refs/heads/main/laptopdata.csv
df = pd.read_csv("laptopdata.csv")

df['is_apple'] = (df['Brand'] == "Apple").astype(int)
df['is_touch'] = (df['Touch'] == "Yes").astype(int)

df = df.dropna(subset=['RAM', 'Storage', 'Screen', 'Final Price'])

df_unnorm = df.copy()

norm_cols = ['RAM', 'Storage', 'Screen', 'Final Price']
means = df_unnorm[norm_cols].mean()
stds = df_unnorm[norm_cols].std()

for col in norm_cols:
    df[col] = (df[col] - means[col]) / stds[col]

print("Data prepped")
print(df[['RAM','Storage','Screen','Final Price','is_apple']].head())

Question 2

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df_unnorm['Final Price'], bins=30, kde=True)
plt.title("Distribution of Laptop Prices")
plt.show()

In [ ]:
cols = ['RAM','Storage','Screen','is_touch','is_apple','Final Price']
plt.figure(figsize=(9,7))
sns.heatmap(df_unnorm[cols].corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df_unnorm, x='RAM', y='Final Price', hue='is_apple')
plt.title("RAM vs Price")
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df_unnorm, x='Storage', y='Final Price', hue='is_apple')
plt.title("Storage vs Price")
plt.show()

Question 3

model 1

In [ ]:
X1 = df[['Storage']].values
y = df['Final Price'].values

indices = np.random.permutation(len(df))
split_idx = int(0.8 * len(df))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X1_train = torch.tensor(X1[train_idx], dtype=torch.float32)
X1_test = torch.tensor(X1[test_idx], dtype=torch.float32)
y_train = torch.tensor(y[train_idx], dtype=torch.float32)
y_test = torch.tensor(y[test_idx], dtype=torch.float32)

params1 = torch.randn(2, requires_grad=True)

learning_rate = 0.05
epochs = 1001

for epoch in range(epochs):

    output = params1[0]*X1_train[:,0] + params1[1]

    loss = torch.mean((output - y_train)**2)

    loss.backward()

    with torch.no_grad():
        params1 -= learning_rate * params1.grad
        params1.grad.zero_()

with torch.no_grad():
    test = params1[0]*X1_test[:,0] + params1[1]
    preds1 = test.numpy()


preds_real = preds1 * stds['Final Price'] + means['Final Price']
y_test_real = y_test.numpy() * stds['Final Price'] + means['Final Price']

mse = np.mean((y_test_real - preds_real)**2)
r2 = np.corrcoef(y_test_real, preds_real)[0,1]**2

print("MSE:", mse)
print("R^2:", r2)
w = params1[0].item()
b = params1[1].item()
print("Final training loss:", loss.item())

w_real = (stds['Final Price'] / stds['Storage']) * w
b_real = means['Final Price'] + stds['Final Price'] * b - w_real * means['Storage']

print(f"model equation:")
print(f"Price = {w_real:.3f} * Storage + {b_real:.3f}")

model 2 multi linear

In [ ]:
df['is_ssd'] = (df['Storage type'] == 'SSD').astype(int)
features_to_use = ['RAM', 'Screen', 'is_ssd']
X2 = df[features_to_use].values
y = df['Final Price'].values

X2_tensor = torch.tensor(X2, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

indices = np.random.permutation(len(df))
X2_tensor = X2_tensor[indices]
y_tensor = y_tensor[indices]

k_folds = 5
fold_size = len(df) // k_folds

mse_scores = []
r2_scores = []
all_params = []
for fold in range(k_folds):

    start = fold * fold_size
    end = (fold + 1) * fold_size

    X_test = X2_tensor[start:end]
    y_test = y_tensor[start:end]

    X_train = torch.cat((X2_tensor[:start], X2_tensor[end:]), dim=0)
    y_train = torch.cat((y_tensor[:start], y_tensor[end:]), dim=0)

    params_cv = torch.randn(4, requires_grad=True)

    for epoch in range(600):
        output = (params_cv[0]*X_train[:,0] +
                  params_cv[1]*X_train[:,1] +
                  params_cv[2]*X_train[:,2] +
                  params_cv[3])

        loss = torch.mean((output - y_train)**2)

        loss.backward()

        with torch.no_grad():
            params_cv -= 0.05 * params_cv.grad
            params_cv.grad.zero_()
    all_params.append(params_cv.detach().numpy())

    with torch.no_grad():
        pred = (params_cv[0]*X_test[:,0] +
                params_cv[1]*X_test[:,1] +
                params_cv[2]*X_test[:,2] +
                params_cv[3]).numpy()

    pred_real = pred * stds['Final Price'] + means['Final Price']
    y_test_real = y_test.numpy() * stds['Final Price'] + means['Final Price']

    mse = np.mean((y_test_real - pred_real)**2)
    r2 = np.corrcoef(y_test_real, pred_real)[0,1]**2

    mse_scores.append(mse)
    r2_scores.append(r2)

    print(f"Fold {fold+1}  MSE: {mse:.2f}, R^2: {r2:.4f}")

avg_params = np.mean(all_params, axis=0)
print(f"Avg MSE: {np.mean(mse_scores):.2f}")
print(f"Avg R^2: {np.mean(r2_scores):.4f}")
print(f"Final training loss:", loss.item())

print("\nModel 2 Equation:")
w1, w2, w3, b = avg_params

w1_real = (stds['Final Price'] / stds['RAM']) * w1
w2_real = (stds['Final Price'] / stds['Screen']) * w2
w3_real = stds['Final Price'] * w3
b_real = means['Final Price'] + stds['Final Price'] * b - w1_real * means['RAM']  - w2_real * means['Screen']

print(f"Price = {w1_real:.3f}*RAM + {w2_real:.3f}*Screen + {w3_real:.3f}*is_ssd + {b_real:.3f}")

Question 4

In [ ]:
features = ['RAM', 'Screen', 'is_ssd']
X = df[features].values
y = df['Final Price'].values
indices = np.random.permutation(len(df))
split_idx = int(0.8 * len(df))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_test = X[test_idx]
y_test = y[test_idx]
w1, w2, w3, b = avg_params
pred = (w1*X_test[:,0] +
        w2*X_test[:,1] +
        w3*X_test[:,2] +
        b)

pred_real = pred * stds['Final Price'] + means['Final Price']
y_real = y_test * stds['Final Price'] + means['Final Price']

is_apple_test = df_unnorm.iloc[test_idx]['is_apple'].values
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_real, y=pred_real, hue=is_apple_test)


plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Price")
plt.show()


In [ ]:
residuals = y_real - pred_real

apple_residuals = residuals[is_apple_test == 1]
nonapple_residuals = residuals[is_apple_test == 0]

print("Apple laptops average residual:", np.mean(apple_residuals))
print("Non-Apple laptops average residual:", np.mean(nonapple_residuals))

Question 5

In [ ]:
features = ['RAM', 'Storage', 'is_ssd', 'is_touch', 'Final Price']
X = df[features].values
y = df['is_apple'].values

indices = np.random.permutation(len(df))
split_idx = int(0.8 * len(df))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train = torch.tensor(X[train_idx], dtype=torch.float32)
X_test = torch.tensor(X[test_idx], dtype=torch.float32)

y_train = torch.tensor(y[train_idx], dtype=torch.float32)
y_test = y[test_idx]

params = torch.randn(6, requires_grad=True)

learning_rate = 0.1
epochs = 1000

for epoch in range(epochs):
  z = (params[0]*X_train[:,0] +
         params[1]*X_train[:,1] +
         params[2]*X_train[:,2] +
         params[3]*X_train[:,3] +
         params[4]*X_train[:,4] +
         params[5])

  pred = torch.sigmoid(z)
  loss = -torch.mean(
    y_train * torch.log(pred + 1e-8) +
    (1 - y_train) * torch.log(1 - pred + 1e-8)
)

  loss.backward()

  with torch.no_grad():
        params -= learning_rate * params.grad
        params.grad.zero_()

  if epoch % 200 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")



In [ ]:
with torch.no_grad():
    z_test = (params[0]*X_test[:,0] +
              params[1]*X_test[:,1] +
              params[2]*X_test[:,2] +
              params[3]*X_test[:,3] +
              params[4]*X_test[:,4] +
              params[5])

    probs = torch.sigmoid(z_test).numpy()
preds = (probs >= 0.1).astype(int)
accuracy = np.mean(preds == y_test)
precision = np.sum((preds == 1) & (y_test == 1)) / max(np.sum(preds == 1), 1)

In [ ]:
print("\nModel Weights:")
print(f"RAM: {params[0].item():.3f}")
print(f"Storage: {params[1].item():.3f}")
print(f"Screen: {params[2].item():.3f}")
print(f"is_touch: {params[3].item():.3f}")
print(f"Final Price: {params[4].item():.3f}")

Extra Credit

In [ ]:
df['RAM_Storage'] = df['RAM'] * df['Storage']
df['RAM_Screen'] = df['RAM'] * df['Screen']
df['Storage_Screen'] = df['Storage'] * df['Screen']

df['RAM2'] = df['RAM']**2
df['Screen2'] = df['Screen']**2
df['Storage2'] = df['Storage']**2

In [ ]:
cols = ['RAM', 'Storage', 'Screen',
        'RAM_Storage', 'RAM_Screen', 'Storage_Screen',
        'RAM2', 'Storage2', 'Screen2','Final Price']

corr = df[cols].corr()

print(corr['Final Price'].sort_values(ascending=False))

In [ ]:
features = ['RAM', 'Storage', 'RAM_Screen', 'RAM2']
X = df[features].values
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X = (X - X_mean) / X_std
y = df['Final Price'].values

indices = np.random.permutation(len(df))
split_idx = int(0.8 * len(df))
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train = torch.tensor(X[train_idx], dtype=torch.float32)
X_test = torch.tensor(X[test_idx], dtype=torch.float32)

y_train = torch.tensor(y[train_idx], dtype=torch.float32)
y_test = y[test_idx]

params = torch.randn(5, requires_grad=True)

learning_rate = 0.01
epochs = 1500

for epoch in range(epochs):

    output = (params[0]*X_train[:,0] +
          params[1]*X_train[:,1] +
          params[2]*X_train[:,2] +
          params[3]*X_train[:,3] +
          params[4])

    loss = torch.mean((output - y_train)**2)

    loss.backward()

    with torch.no_grad():
        params -= learning_rate * params.grad
        params.grad.zero_()

    if epoch % 300 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")


In [ ]:
with torch.no_grad():

    pred = (params[0]*X_test[:,0] +
        params[1]*X_test[:,1] +
        params[2]*X_test[:,2] +
        params[3]*X_test[:,3] +
        params[4]).numpy()

pred_real = pred * stds['Final Price'] + means['Final Price']
y_test_real = y_test * stds['Final Price'] + means['Final Price']

mse = np.mean((y_test_real - pred_real)**2)
r2 = np.corrcoef(y_test_real, pred_real)[0,1]**2

print("MSE:", mse)
print("R^2:", r2)

In [ ]:
w1, w2, w3, w4, b = params.detach().numpy()
w1_real = (stds['Final Price'] / stds['RAM']) * w1
w2_real = (stds['Final Price'] / stds['Storage']) * w2
w3_real = stds['Final Price']* w3
w4_real = stds['Final Price']* w4
b_real = means['Final Price'] + stds['Final Price'] * b - w1_real * means['RAM'] - w2_real * means['Storage']

print(f"Price = {w1_real:.3f}*RAM + {w2_real:.3f}*Storage + {w3_real:.3f}*RAM_Screen + {w4_real:.3f}*RAM^2 + {b_real:.3f}")